# Mi primer y fantastico notebook en Colab: análisis de ventas

En este notebook vamos a:
1. Comprobar el entorno en el que estamos trabajando
2. Generar un pequeño dataset de ventas
3. Explorarlo y calcular algunas métricas con pandas
4. Dibujar un gráfico
5. Guardar el resultado

Ejecuta cada celda con **Shift+Enter**.

## 1. Comprobar el entorno

In [ ]:
import sys
import pandas as pd
import numpy as np

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

Los comandos de terminal funcionan con `!` delante:

In [ ]:
!ls -la

## 2. Generar un dataset de ventas

Creamos 500 ventas inventadas de una cadena de cafeterías. Fijamos la semilla (`seed`) para que todos obtengamos los mismos datos.

In [ ]:
rng = np.random.default_rng(seed=42)
n = 500

precios = {"Café": 1.50, "Té": 1.80, "Zumo": 2.50, "Bocadillo": 4.00}

ventas = pd.DataFrame({
    "fecha": pd.to_datetime("2026-01-01") + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
    "tienda": rng.choice(["Madrid", "Barcelona", "A Coruña", "Sevilla"], n),
    "producto": rng.choice(list(precios), n, p=[0.4, 0.15, 0.2, 0.25]),
    "unidades": rng.integers(1, 6, n),
})
ventas["precio"] = ventas["producto"].map(precios)

ventas.head()

## 3. Explorar los datos

In [ ]:
print("Filas y columnas:", ventas.shape)
ventas.dtypes

In [ ]:
ventas.describe()

¿Cuántas veces se ha vendido cada producto? Es lo mismo que hacíamos en la terminal con `sort | uniq -c`:

In [ ]:
ventas["producto"].value_counts()

Calculamos el importe de cada venta. Fíjate en que no hay ningún bucle `for`: la operación es **vectorizada** y la ejecuta numpy por dentro, en código compilado.

In [ ]:
ventas["importe"] = ventas["unidades"] * ventas["precio"]
ventas.head()

Facturación por tienda, de mayor a menor:

In [ ]:
por_tienda = (
    ventas.groupby("tienda")["importe"]
    .sum()
    .sort_values(ascending=False)
    .round(2)
)
por_tienda

## 4. Un gráfico: facturación mensual por tienda

In [ ]:
import matplotlib.pyplot as plt

mensual = (
    ventas.assign(mes=ventas["fecha"].dt.to_period("M"))
    .pivot_table(index="mes", columns="tienda", values="importe", aggfunc="sum")
)

ax = mensual.plot(kind="bar", figsize=(10, 5), width=0.8)
ax.set_title("Facturación mensual por tienda")
ax.set_xlabel("Mes")
ax.set_ylabel("Euros")
ax.legend(title="Tienda")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Guardar el resultado

El archivo se guarda en el disco de la sesión y **se borra al cerrarla**. Lo verás en el panel de la carpeta, a la izquierda, y desde ahí puedes descargarlo.

In [ ]:
ventas.to_csv("ventas.csv", index=False)
print("Guardado ventas.csv")

Para que no se pierda, guárdalo en tu Drive. La primera vez, Colab te pedirá permiso para acceder a él.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ventas.to_csv("/content/drive/MyDrive/ventas.csv", index=False)
    print("Guardado en tu Drive: MyDrive/ventas.csv")
except ImportError:
    print("No estás en Colab: esta celda solo funciona allí")

## Ejercicio

1. ¿Qué producto factura más en total? Pista: igual que `por_tienda`, pero agrupando por `"producto"`.
2. ¿Qué día de la semana se vende más? Pista: `ventas["fecha"].dt.day_name()`.
3. Añade una celda de texto con tus conclusiones.